# Comparison **FCM**, **GMM**, and **Spectral Clustering**.\n

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Comparison Code with Updated Proposed Method

Updated proposed part:
- Inner hyper-ellipsoid = fuzzy center nucleus
- Outer hyper-ellipsoid = Type-2 uncertainty boundary
- Assignment uses minimum Mahalanobis-type distance to the fuzzy-center nucleus:
    D_j(x) = min_{z in N_j} (x-z)^T Sigma_j^{-1} (x-z)

For the ellipsoidal nucleus:
    q_j(x) = (x-c_j)^T Sigma_j^{-1} (x-c_j)

    If q_j(x) <= tau_L_j:
        D_j(x) = 0
    Else:
        D_j(x) = (sqrt(q_j(x)) - sqrt(tau_L_j))^2

Output:
Result_<dataset_name>/comparison_<dataset_name>.csv
"""

import time
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

# ============================================================
# USER SETTINGS
# ============================================================

DATASET_PATH = "your_dataset.csv"
N_CLUSTERS = 3
LABEL_COLUMN = None          # Example: "target"; keep None if no class label

RANDOM_STATE = 42
MAX_ITER = 100
EPSILON = 1e-4

NUCLEUS_QUANTILE = 0.40
TYPE2_QUANTILE = 0.80


# ============================================================
# DATA LOADING AND PREPROCESSING
# ============================================================

def load_dataset(dataset_path, label_column=None):
    df = pd.read_csv(dataset_path)

    for col in df.columns:
        if df[col].dtype == "object" or df[col].dtype.name == "category":
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    y_true = None
    if label_column is not None and label_column in df.columns:
        y_true = df[label_column].values
        X_df = df.drop(columns=[label_column])
    else:
        X_df = df.copy()

    X_df = X_df.select_dtypes(include=[np.number])
    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    X_df = X_df.fillna(X_df.median(numeric_only=True))

    scaler = StandardScaler()
    X = scaler.fit_transform(X_df.values)

    return X, y_true, X_df


# ============================================================
# METRIC FUNCTION
# ============================================================

def compute_metrics(X, labels, y_true=None):
    results = {}

    if len(np.unique(labels)) > 1:
        results["Silhouette"] = silhouette_score(X, labels)
        results["Davies_Bouldin"] = davies_bouldin_score(X, labels)
        results["Calinski_Harabasz"] = calinski_harabasz_score(X, labels)
    else:
        results["Silhouette"] = np.nan
        results["Davies_Bouldin"] = np.nan
        results["Calinski_Harabasz"] = np.nan

    if y_true is not None:
        results["ARI"] = adjusted_rand_score(y_true, labels)
        results["NMI"] = normalized_mutual_info_score(y_true, labels)
    else:
        results["ARI"] = np.nan
        results["NMI"] = np.nan

    return results


# ============================================================
# FUZZY C-MEANS IMPLEMENTATION
# ============================================================

def fuzzy_c_means(X, n_clusters, m=2.0, max_iter=100, error=1e-5, random_state=42):
    np.random.seed(random_state)
    N = X.shape[0]

    U = np.random.rand(N, n_clusters)
    U = U / np.sum(U, axis=1, keepdims=True)

    for _ in range(max_iter):
        U_old = U.copy()

        um = U ** m
        centers = (um.T @ X) / np.sum(um.T, axis=1, keepdims=True)

        dist = np.zeros((N, n_clusters))
        for j in range(n_clusters):
            dist[:, j] = np.linalg.norm(X - centers[j], axis=1)

        dist = np.fmax(dist, 1e-10)

        for j in range(n_clusters):
            U[:, j] = 1.0 / np.sum(
                (dist[:, j:j+1] / dist) ** (2 / (m - 1)),
                axis=1
            )

        if np.linalg.norm(U - U_old) < error:
            break

    labels = np.argmax(U, axis=1)
    return labels, centers, U


# ============================================================
# UPDATED PROPOSED METHOD
# ============================================================

def compute_covariance(X_cluster, center, epsilon=1e-4):
    d = X_cluster.shape[1]

    if X_cluster.shape[0] <= 1:
        return np.eye(d) + epsilon * np.eye(d)

    centered = X_cluster - center
    cov = (centered.T @ centered) / (X_cluster.shape[0] - 1)

    return cov + epsilon * np.eye(d)


def mahalanobis_q_distance(X, centers, covariances):
    N = X.shape[0]
    K = centers.shape[0]
    Q = np.zeros((N, K))

    for j in range(K):
        diff = X - centers[j]
        inv_cov = np.linalg.pinv(covariances[j])
        Q[:, j] = np.sum((diff @ inv_cov) * diff, axis=1)

    return Q


def nucleus_region_distance(X, centers, covariances, tau_L):
    Q = mahalanobis_q_distance(X, centers, covariances)
    D = np.zeros_like(Q)

    for j in range(centers.shape[0]):
        tau = tau_L[j]
        if not np.isfinite(tau) or tau <= 0:
            D[:, j] = Q[:, j]
        else:
            D[:, j] = np.maximum(
                0.0,
                np.sqrt(np.maximum(Q[:, j], 0.0)) - np.sqrt(tau)
            ) ** 2

    return D, Q


def estimate_tau_values(X, labels, centers, covariances, n_clusters):
    Q = mahalanobis_q_distance(X, centers, covariances)

    tau_L = np.zeros(n_clusters)
    tau_U = np.zeros(n_clusters)

    for j in range(n_clusters):
        idx = np.where(labels == j)[0]
        qj = Q[idx, j]

        if len(qj) == 0:
            tau_L[j] = np.nan
            tau_U[j] = np.nan
        else:
            tau_L[j] = np.quantile(qj, NUCLEUS_QUANTILE)
            tau_U[j] = np.quantile(qj, TYPE2_QUANTILE)

            if tau_U[j] <= tau_L[j]:
                tau_U[j] = tau_L[j] + EPSILON

    return tau_L, tau_U, Q


def proposed_it2_kmeans_nucleus_center(
    X,
    n_clusters,
    max_iter=100,
    epsilon=1e-4,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=10
    )

    labels = kmeans.fit_predict(X)
    centers = kmeans.cluster_centers_.copy()

    covariances = np.array([
        compute_covariance(X[labels == j], centers[j], epsilon)
        for j in range(n_clusters)
    ])

    tau_L, tau_U, Q = estimate_tau_values(X, labels, centers, covariances, n_clusters)

    for _ in range(max_iter):
        old_centers = centers.copy()
        old_labels = labels.copy()

        covariances = []
        for j in range(n_clusters):
            Xj = X[labels == j]

            if Xj.shape[0] == 0:
                rand_idx = rng.integers(0, X.shape[0])
                centers[j] = X[rand_idx]
                Xj = X[[rand_idx]]

            centers[j] = Xj.mean(axis=0)
            covariances.append(compute_covariance(Xj, centers[j], epsilon))

        covariances = np.array(covariances)

        tau_L, tau_U, Q = estimate_tau_values(X, labels, centers, covariances, n_clusters)

        D_region, Q = nucleus_region_distance(X, centers, covariances, tau_L)
        labels = np.argmin(D_region, axis=1)

        shift = np.linalg.norm(centers - old_centers)
        label_changes = np.sum(labels != old_labels)

        if shift < 1e-5 and label_changes == 0:
            break

    covariances = []
    for j in range(n_clusters):
        Xj = X[labels == j]
        if Xj.shape[0] == 0:
            rand_idx = rng.integers(0, X.shape[0])
            centers[j] = X[rand_idx]
            Xj = X[[rand_idx]]
        else:
            centers[j] = Xj.mean(axis=0)
        covariances.append(compute_covariance(Xj, centers[j], epsilon))

    covariances = np.array(covariances)
    tau_L, tau_U, Q = estimate_tau_values(X, labels, centers, covariances, n_clusters)
    D_region, Q = nucleus_region_distance(X, centers, covariances, tau_L)

    nd_values = []
    tc_values = []
    outside_values = []

    for j in range(n_clusters):
        idx = np.where(labels == j)[0]
        qj = Q[idx, j]

        if len(qj) == 0:
            nd_values.append(np.nan)
            tc_values.append(np.nan)
            outside_values.append(np.nan)
            continue

        nucleus_count = np.sum(qj <= tau_L[j])
        type2_count = np.sum(qj <= tau_U[j])
        outside_count = np.sum(qj > tau_U[j])

        nd_values.append(nucleus_count / len(qj))
        tc_values.append(type2_count / len(qj))
        outside_values.append(outside_count / len(qj))

    avg_nd = np.nanmean(nd_values)
    avg_tc = np.nanmean(tc_values)
    avg_outside = np.nanmean(outside_values)

    return labels, centers, avg_nd, avg_tc, avg_outside


# ============================================================
# MAIN COMPARISON
# ============================================================

def main():
    dataset_name = Path(DATASET_PATH).stem
    result_dir = Path(f"Result_{dataset_name}")
    result_dir.mkdir(exist_ok=True)

    X, y_true, X_df = load_dataset(DATASET_PATH, LABEL_COLUMN)

    comparison_results = []

    start = time.time()

    labels, centers, avg_nd, avg_tc, avg_outside = proposed_it2_kmeans_nucleus_center(
        X,
        n_clusters=N_CLUSTERS,
        max_iter=MAX_ITER,
        epsilon=EPSILON,
        random_state=RANDOM_STATE
    )

    runtime = time.time() - start
    metrics = compute_metrics(X, labels, y_true)

    comparison_results.append({
        "Method": "Proposed IT2 Nucleus-Center K-means",
        "Assignment": "Minimum distance to fuzzy-center nucleus",
        "Silhouette": metrics["Silhouette"],
        "Davies_Bouldin": metrics["Davies_Bouldin"],
        "Calinski_Harabasz": metrics["Calinski_Harabasz"],
        "ARI": metrics["ARI"],
        "NMI": metrics["NMI"],
        "Avg_Nucleus_Density": avg_nd,
        "Avg_Type2_Coverage": avg_tc,
        "Avg_Outside_Type2_Ratio": avg_outside,
        "Runtime_Seconds": runtime,
        "Time_Complexity": "O(T N K d^2)"
    })

    start = time.time()

    labels, centers, membership = fuzzy_c_means(
        X,
        n_clusters=N_CLUSTERS,
        max_iter=MAX_ITER,
        random_state=RANDOM_STATE
    )

    runtime = time.time() - start
    metrics = compute_metrics(X, labels, y_true)

    comparison_results.append({
        "Method": "Fuzzy C-means",
        "Assignment": "Maximum membership",
        "Silhouette": metrics["Silhouette"],
        "Davies_Bouldin": metrics["Davies_Bouldin"],
        "Calinski_Harabasz": metrics["Calinski_Harabasz"],
        "ARI": metrics["ARI"],
        "NMI": metrics["NMI"],
        "Avg_Nucleus_Density": np.nan,
        "Avg_Type2_Coverage": np.nan,
        "Avg_Outside_Type2_Ratio": np.nan,
        "Runtime_Seconds": runtime,
        "Time_Complexity": "O(T N K d)"
    })

    start = time.time()

    gmm = GaussianMixture(
        n_components=N_CLUSTERS,
        covariance_type="full",
        random_state=RANDOM_STATE
    )

    labels = gmm.fit_predict(X)

    runtime = time.time() - start
    metrics = compute_metrics(X, labels, y_true)

    comparison_results.append({
        "Method": "Gaussian Mixture Model",
        "Assignment": "Maximum posterior probability",
        "Silhouette": metrics["Silhouette"],
        "Davies_Bouldin": metrics["Davies_Bouldin"],
        "Calinski_Harabasz": metrics["Calinski_Harabasz"],
        "ARI": metrics["ARI"],
        "NMI": metrics["NMI"],
        "Avg_Nucleus_Density": np.nan,
        "Avg_Type2_Coverage": np.nan,
        "Avg_Outside_Type2_Ratio": np.nan,
        "Runtime_Seconds": runtime,
        "Time_Complexity": "O(T N K d^2)"
    })

    start = time.time()

    spectral = SpectralClustering(
        n_clusters=N_CLUSTERS,
        affinity="nearest_neighbors",
        random_state=RANDOM_STATE,
        assign_labels="kmeans"
    )

    labels = spectral.fit_predict(X)

    runtime = time.time() - start
    metrics = compute_metrics(X, labels, y_true)

    comparison_results.append({
        "Method": "Spectral Clustering",
        "Assignment": "Graph partition + K-means",
        "Silhouette": metrics["Silhouette"],
        "Davies_Bouldin": metrics["Davies_Bouldin"],
        "Calinski_Harabasz": metrics["Calinski_Harabasz"],
        "ARI": metrics["ARI"],
        "NMI": metrics["NMI"],
        "Avg_Nucleus_Density": np.nan,
        "Avg_Type2_Coverage": np.nan,
        "Avg_Outside_Type2_Ratio": np.nan,
        "Runtime_Seconds": runtime,
        "Time_Complexity": "Approximately O(N^3)"
    })

    comparison_df = pd.DataFrame(comparison_results)

    output_file = result_dir / f"comparison_{dataset_name}.csv"
    comparison_df.to_csv(output_file, index=False)

    print("\nComparison Results:")
    print(comparison_df)

    print(f"\nSaved comparison table at: {output_file}")


if __name__ == "__main__":
    main()


# Added K-Means and DBNS
DBNS = Silhouette / Davies_Bouldin.
Add a Simple K-Means method and a DBNS column to all result dictionaries.
